# Домашнє завдання 3. Завантаження, типізація та аудит даних у PostgreSQL

Mykyta Oliinyk

Дисципліна: Relational Databases for Data Science. Датасет: NYC TLC Yellow Taxi Trip Records, файл за жовтень 2025 року (офіційний Parquet), контрольована вибірка 100 000 поїздок, плюс довідник зон TLC.

Маршрут роботи: Parquet → CSV → `COPY FROM STDIN` у staging (`TEXT`) → `INSERT INTO ... SELECT` у типізовану таблицю → аудит якості → розподіли й аномалії → 12 DQL/EDA-запитів → Reflection.

## Завдання 1. Setup і завантаження

### 1.1. Запуск PostgreSQL через pgserver

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo "psycopg[binary]"

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 11.8 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Два уточнення до підключення. Обидва потрібні, щоб notebook працював однаково в Colab і локально.

- **Драйвер.** SQLAlchemy 2.1 для адреси `postgresql://` обирає psycopg 3, а SQLAlchemy 2.0 у Colab обирає psycopg2. Метод `copy_expert` із завдання є лише в psycopg2, тому драйвер задано явно.
- **Часовий пояс сесії.** Значення `TIMESTAMPTZ` виводиться в поясі сесії, а він за замовчуванням залежить від машини. Фіксуємо `America/New_York`, щоб результати не залежали від місця запуску.

In [2]:
from sqlalchemy.engine import make_url

engine.dispose()
engine = create_engine(
    make_url(pg.get_uri()).set(drivername='postgresql+psycopg2'),
    future=True,
    connect_args={'options': '-c timezone=America/New_York'},
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)


def run(sql):
    # DDL і DML у межах однієї транзакції
    with engine.begin() as conn:
        conn.execute(text(sql))


def q(sql):
    # SELECT із поверненням DataFrame; час показуємо в поясі Нью-Йорка
    df = pd.read_sql(text(sql), engine)
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_convert('America/New_York')
    return df


q("SELECT current_setting('server_version') AS server_version, current_setting('TimeZone') AS session_time_zone")

,server_version,session_time_zone
0,16.2,America/New_York


### 1.2. Отримання даних з офіційного джерела

Джерело: сторінка [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), файл Yellow Taxi за 2025-10. У цьому місяці немає переходу на літній чи зимовий час (він відбувся 2 листопада), тому локальний час у файлі однозначно перетворюється на момент часу.

Завдання просить зберегти лише потрібні колонки, і тут потрібні всі 20. До 11 колонок із прикладу додано `RatecodeID` для аналізу договірних тарифів, сім складових чека для перевірки `total_amount` у запиті 6 і `store_and_fwd_flag`, єдину справжню двійкову ознаку файлу, потрібну для типу `BOOLEAN`. Назва `Airport_fee` у цьому файлі написана з великої літери.

Вибірка зберігається в `data/hw3_taxi_sample.csv`: її розмір менший за ліміт 100 МБ, тому CSV входить до репозиторію разом із довідником зон. Місячний Parquet потрібен лише для формування вибірки, тому він завантажується в тимчасову папку системи й до репозиторію не потрапляє.

In [3]:
import tempfile
from pathlib import Path
from urllib.request import urlretrieve

import pyarrow as pa
import pyarrow.parquet as pq

DATA_MONTH = '2025-10'
DATA_URL = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{DATA_MONTH}.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
SAMPLE_SIZE = 100_000
RANDOM_SEED = 42

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
parquet_path = Path(tempfile.gettempdir()) / f'hw3_taxi_{DATA_MONTH}.parquet'
zones_path = DATA_DIR / 'hw3_taxi_zones.csv'
csv_path = DATA_DIR / 'hw3_taxi_sample.csv'

if not parquet_path.exists():
    urlretrieve(DATA_URL, parquet_path)
if not zones_path.exists():
    urlretrieve(ZONES_URL, zones_path)

columns = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
    'PULocationID', 'DOLocationID', 'payment_type',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'total_amount',
    'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
]
parquet_meta = pq.read_metadata(parquet_path)
print('Рядків у файлі за місяць:', parquet_meta.num_rows)
print('Колонок:', parquet_meta.num_columns)


Рядків у файлі за місяць: 4428699
Колонок: 20


### 1.3. Як сформовано вибірку

Приклад із завдання бере `.head(100_000)`. Для цього файлу такий зріз непридатний, бо рядки у файлі впорядковані не випадково. Порівнюємо його з випадковою вибіркою з фіксованим seed. Весь файл у пам'яті не тримаємо: для порівняння й перевірок читаємо лише дев'ять колонок, а `sample` обирає рядки за їхніми номерами, тому склад вибірки від кількості колонок не залежить.

In [4]:
audit_columns = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
    'payment_type', 'total_amount',
]
month_df = pd.read_parquet(parquet_path, columns=audit_columns)

sample_df = month_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).sort_index()
head_df = month_df.head(SAMPLE_SIZE)


def describe_slice(df):
    return {
        'n_rows': len(df),
        'first_pickup': df['tpep_pickup_datetime'].min(),
        'last_pickup': df['tpep_pickup_datetime'].max(),
        'n_pickup_dates': df['tpep_pickup_datetime'].dt.date.nunique(),
        'pct_null_passenger_count': round(df['passenger_count'].isna().mean() * 100, 2),
        'pct_payment_type_0': round((df['payment_type'] == 0).mean() * 100, 2),
    }


pd.DataFrame({
    'full_month': describe_slice(month_df),
    'head_100k': describe_slice(head_df),
    'random_sample': describe_slice(sample_df),
}).T

,n_rows,first_pickup,last_pickup,n_pickup_dates,pct_null_passenger_count,pct_payment_type_0
full_month,4428699,2025-09-30 22:54:51,2025-11-01 00:32:12,33,22.37,22.37
head_100k,100000,2025-09-30 22:54:51,2025-10-01 22:53:20,2,0.00,0.00
random_sample,100000,2025-10-01 00:00:07,2025-10-31 23:59:52,31,22.56,22.56


**Висновок.** `head(100_000)` охоплює лише одну добу: з вечора 30 вересня до вечора 1 жовтня. Рядків із `payment_type = 0` і `NULL` у `passenger_count` у цьому зрізі немає взагалі (0,00%), хоча у файлі їх 22,37%. Аудит якості на такому зрізі пропустив би головну проблему файлу, а розподіл за днями був би неможливий.

Тому вибірка формується так: `month_df.sample(n=100_000, random_state=42).sort_index()`. Це 2,26% файлу з 4 428 699 рядків. Вибірка покриває всі 31 дату жовтня, частка `NULL` у ній становить 22,56%, а фіксований seed робить її відтворюваною. Усі подальші числа стосуються вибірки, а не всього місяця.

Деякі перевірки має сенс порахувати до вибірки, бо випадковий зріз їх майже не зберігає: рідкісні помилки часу та пари рядків, які мають потрапити у вибірку обидва. Ці числа використано далі в аудиті. Повторення ключа спершу шукаємо за 64-бітним хешем, бо це потребує значно менше пам'яті. Точний дублікат завжди має той самий ключ, тому всі 20 колонок читаємо лише для рядків із повтореним ключем і для рядків вибірки, по одній row group за раз.

In [5]:
def read_rows(path, row_numbers, column_names):
    # Читає файл частинами й залишає лише потрібні рядки
    parquet_file = pq.ParquetFile(path)
    parts = []
    start = 0
    for group_id in range(parquet_file.num_row_groups):
        group = parquet_file.read_row_group(group_id, columns=column_names)
        stop = start + group.num_rows
        rows = row_numbers[(row_numbers >= start) & (row_numbers < stop)]
        parts.append(group.take(rows - start))
        start = stop
    return pa.concat_tables(parts).to_pandas()


month_start = pd.Timestamp(f'{DATA_MONTH}-01')
month_end = month_start + pd.offsets.MonthBegin(1)
pickup = month_df['tpep_pickup_datetime']
dropoff = month_df['tpep_dropoff_datetime']
trip_key = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_distance',
]
key_hash = pd.util.hash_pandas_object(month_df[trip_key], index=False)
candidates = month_df[key_hash.duplicated(keep=False)]
same_trip = candidates[candidates.duplicated(trip_key, keep=False)]
pair_sums = same_trip.groupby(trip_key, sort=False)['total_amount'].sum().round(2)

month_checks = {
    'pickup_outside_month': int(((pickup < month_start) | (pickup >= month_end)).sum()),
    'dropoff_before_pickup': int((dropoff < pickup).sum()),
    'rows_sharing_trip_key': len(same_trip),
    'groups_sharing_trip_key': len(pair_sums),
    'groups_with_zero_total': int((pair_sums == 0).sum()),
}
same_trip_rows = same_trip.index.to_numpy()
sample_rows = sample_df.index.to_numpy()
del month_df, head_df, sample_df, pickup, dropoff, key_hash, candidates, same_trip, pair_sums

same_trip_full = read_rows(parquet_path, same_trip_rows, columns)
month_checks['exact_duplicate_rows'] = int(same_trip_full.duplicated().sum())
del same_trip_full

pd.Series(month_checks, name='full_month').to_frame()


,full_month
pickup_outside_month,13
dropoff_before_pickup,2
rows_sharing_trip_key,122978
groups_sharing_trip_key,61489
groups_with_zero_total,61481
exact_duplicate_rows,0


**Висновок.** У всьому файлі 13 рядків мають подачу поза жовтнем, 2 рядки мають висадку раніше подачі, а точних дублікатів немає. Натомість 122 978 рядків утворюють 61 489 пар із тим самим часом, зонами й відстанню, і в 61 481 парі сума `total_amount` дорівнює нулю. Отже, це не повторний запис, а пари нарахування й сторно.

In [6]:
sample_df = read_rows(parquet_path, sample_rows, columns)
sample_df.to_csv(csv_path, index=False)
print('CSV:', csv_path.as_posix(), '| рядків:', len(sample_df), '| МБ:', round(csv_path.stat().st_size / 1024 / 1024, 1))
del sample_df


CSV: data/hw3_taxi_sample.csv | рядків: 100000 | МБ: 10.3


### 1.4. Staging-таблиці та COPY FROM STDIN

У staging усі колонки мають тип `TEXT`: файл завантажується без падіння, навіть якщо в ньому є неочікувані значення, а типізація й очищення відбуваються окремим кроком. Назви колонок відповідають порядку колонок CSV.

Довідник зон потрібен як окрема таблиця: `location_id` у поїздках є кодом, а текстові назви зон і районів є лише в цьому файлі TLC.

In [7]:
run('''
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones_staging CASCADE;

CREATE TABLE hw3_taxi_staging (
    vendor_id_raw             TEXT,
    pickup_raw                TEXT,
    dropoff_raw               TEXT,
    passenger_count_raw       TEXT,
    trip_distance_raw         TEXT,
    ratecode_id_raw           TEXT,
    store_and_fwd_flag_raw    TEXT,
    pu_location_id_raw        TEXT,
    do_location_id_raw        TEXT,
    payment_type_raw          TEXT,
    fare_amount_raw           TEXT,
    extra_raw                 TEXT,
    mta_tax_raw               TEXT,
    tip_amount_raw            TEXT,
    tolls_amount_raw          TEXT,
    improvement_surcharge_raw TEXT,
    total_amount_raw          TEXT,
    congestion_surcharge_raw  TEXT,
    airport_fee_raw           TEXT,
    cbd_congestion_fee_raw    TEXT
);

CREATE TABLE hw3_taxi_zones_staging (
    location_id_raw  TEXT,
    borough_raw      TEXT,
    zone_raw         TEXT,
    service_zone_raw TEXT
);
''')

In [8]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                '''
                COPY hw3_taxi_staging (
                    vendor_id_raw,
                    pickup_raw,
                    dropoff_raw,
                    passenger_count_raw,
                    trip_distance_raw,
                    ratecode_id_raw,
                    store_and_fwd_flag_raw,
                    pu_location_id_raw,
                    do_location_id_raw,
                    payment_type_raw,
                    fare_amount_raw,
                    extra_raw,
                    mta_tax_raw,
                    tip_amount_raw,
                    tolls_amount_raw,
                    improvement_surcharge_raw,
                    total_amount_raw,
                    congestion_surcharge_raw,
                    airport_fee_raw,
                    cbd_congestion_fee_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                ''',
                file,
            )
        with open(zones_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                '''
                COPY hw3_taxi_zones_staging (
                    location_id_raw,
                    borough_raw,
                    zone_raw,
                    service_zone_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                ''',
                file,
            )
    raw_conn.commit()
finally:
    raw_conn.close()

q('''
SELECT 'hw3_taxi_staging' AS table_name, COUNT(*) AS loaded_rows FROM hw3_taxi_staging
UNION ALL
SELECT 'hw3_taxi_zones_staging', COUNT(*) FROM hw3_taxi_zones_staging
''')

,table_name,loaded_rows
0,hw3_taxi_staging,100000
1,hw3_taxi_zones_staging,265


**Висновок.** У `hw3_taxi_staging` завантажено 100 000 рядків, що дорівнює кількості рядків CSV без заголовка. У `hw3_taxi_zones_staging` завантажено 265 зон.

## Завдання 2. Staging і DDL

### 2.1. Обґрунтування типів і правила очищення

Коди постачальників, тарифів і оплат подано за офіційним [словником TLC](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) у редакції від 18 березня 2025 року. Для всіх колонок спільне правило: `NULLIF(TRIM(value), '')`, тобто порожній рядок перетворюється на `NULL` до приведення типу.

**Таблиця `hw3_taxi_trips` (одна поїздка = один рядок):**

| Колонка | Семантика | Джерельний формат | Тип | Правило очищення |
|---|---|---|---|---|
| `trip_id` | технічний ключ, у файлі природного ключа немає | немає | `BIGINT IDENTITY PRIMARY KEY` | генерується під час вставки |
| `vendor_id` | постачальник системи запису: 1 Creative Mobile Technologies, 2 Curb Mobility, 6 Myle Technologies, 7 Helix | int32, `2` | `SMALLINT NOT NULL`, `CHECK IN (1, 2, 6, 7)` | приведення типу; це код, а не кількість, тому арифметика до нього не застосовується |
| `pickup_at`, `dropoff_at` | моменти, коли таксометр увімкнено й вимкнено | timestamp без поясу, локальний час Нью-Йорка, `2025-10-01 00:15:32` | `TIMESTAMPTZ NOT NULL` | `::TIMESTAMP AT TIME ZONE 'America/New_York'`; рядки з `dropoff_at < pickup_at` не переносяться |
| `pickup_date` | календарна дата подачі в Нью-Йорку | немає, похідна | `DATE`, `GENERATED ALWAYS ... STORED` | виводиться з `pickup_at` у поясі Нью-Йорка, тому не може розійтися з ним і не залежить від пояса сесії |
| `passenger_count` | кількість пасажирів у машині | int64, через `NaN` у CSV стає `1.0` | `SMALLINT`, `CHECK >= 0` | `::NUMERIC::SMALLINT`, бо пряме приведення `'1.0'` до `SMALLINT` дає помилку; `NULL` і `0` зберігаються без заміни |
| `trip_distance` | відстань за таксометром, милі | double, `17.2` | `NUMERIC(10,2) NOT NULL`, `CHECK >= 0` | округлення до сотих; великі значення не видаляються, а розглядаються в 4.3 |
| `ratecode_id` | підсумковий код тарифу наприкінці поїздки: 1 стандартний, 2 JFK, 3 Newark, 4 Nassau або Westchester, 5 договірний, 6 груповий, 99 порожній або невідомий | int64, через `NaN` стає `1.0` | `SMALLINT`, `CHECK IN (1, 2, 3, 4, 5, 6, 99)` | `::NUMERIC::SMALLINT`; код 99 зберігається, бо це значення джерела |
| `store_and_fwd` | чи запис зберігався в машині перед передачею через відсутність зв'язку | рядок `Y` або `N` | `BOOLEAN` | `Y` → `TRUE`, `N` → `FALSE`, інше → `NULL` |
| `pu_location_id`, `do_location_id` | зони подачі й висадки TLC | int32, `132` | `INTEGER NOT NULL`, `FOREIGN KEY` на `hw3_taxi_zones` | приведення типу; зовнішній ключ не допускає неіснуючих зон |
| `payment_type` | спосіб оплати: 0 Flex Fare, 1 картка, 2 готівка, 3 без оплати, 4 спір, 5 невідомо, 6 скасовано | int64, `1` | `SMALLINT NOT NULL`, `CHECK BETWEEN 0 AND 6` | приведення типу, код зберігається; підписи додаються в запитах |
| `fare_amount`, `total_amount` | тариф таксометра за час і відстань; підсумкова сума з пасажира без готівкових чайових, доларів | double, `70.0` | `NUMERIC(10,2) NOT NULL` | гроші потребують точності до центів без похибки `double`; від'ємні значення залишаються, бо це сторно |
| `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `cbd_congestion_fee` | складові чека, доларів: `extra` інші доплати, `mta_tax` податок MTA, `tip_amount` чайові карткою, `tolls_amount` платні дороги, `improvement_surcharge` надбавка, що стягується з 2015 року, `cbd_congestion_fee` плата за поїздку в зоні Congestion Relief MTA з 5 січня 2025 року | double, `0.75` | `NUMERIC(10,2)` | ті самі правила, що для грошей; `0` є значенням, а не пропуском |
| `congestion_surcharge`, `airport_fee` | `congestion_surcharge` збір штату Нью-Йорк за затори, `airport_fee` збір за подачу лише в LaGuardia і JFK, доларів; обидва порожні у рядках із `payment_type = 0` | double або порожньо | `NUMERIC(10,2)` | `NULL` зберігається, бо заміна на `0` стверджувала б, що збору не було |

**Таблиця `hw3_taxi_zones` (довідник):**

| Колонка | Семантика | Джерельний формат | Тип | Правило очищення |
|---|---|---|---|---|
| `location_id` | код зони TLC | рядок CSV `132` | `INTEGER PRIMARY KEY` | приведення типу |
| `borough`, `zone`, `service_zone` | район, зона, тип зони | текст у лапках | `TEXT` | бізнес-ліміту довжини немає, тому не `VARCHAR(255)`; маркери `Unknown` і `N/A` → `NULL` |

Застосування вимог до DDL: `NUMERIC(p,s)` для грошей, `DATE` для `pickup_date`, `TIMESTAMPTZ` для `pickup_at` і `dropoff_at`, `BOOLEAN` для `store_and_fwd`, `TEXT` для назв зон. `NOT NULL` мають постачальник, час, зони, спосіб оплати, відстань, `fare_amount` і `total_amount`: без них рядок не описує поїздку. Предметні `CHECK` обмежують коди словником TLC, забороняють від'ємні відстань і кількість пасажирів, а також висадку раніше подачі. Суми без `CHECK` на знак, бо сторно від'ємні.

### 2.2. DDL типізованих таблиць

In [9]:
run('''
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones CASCADE;

CREATE TABLE hw3_taxi_zones (
    location_id  INTEGER PRIMARY KEY,
    borough      TEXT,
    zone         TEXT,
    service_zone TEXT
);

CREATE TABLE hw3_taxi_trips (
    trip_id               BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,
    vendor_id             SMALLINT NOT NULL
        CONSTRAINT chk_trips_vendor_id CHECK (vendor_id IN (1, 2, 6, 7)),
    pickup_at             TIMESTAMPTZ NOT NULL,
    dropoff_at            TIMESTAMPTZ NOT NULL,
    pickup_date           DATE
        GENERATED ALWAYS AS (
            (pickup_at AT TIME ZONE 'America/New_York')::DATE
        ) STORED,
    passenger_count       SMALLINT
        CONSTRAINT chk_trips_passenger_count CHECK (
            passenger_count IS NULL
            OR passenger_count >= 0
        ),
    trip_distance         NUMERIC(10, 2) NOT NULL
        CONSTRAINT chk_trips_distance CHECK (trip_distance >= 0),
    ratecode_id           SMALLINT
        CONSTRAINT chk_trips_ratecode CHECK (
            ratecode_id IS NULL
            OR ratecode_id IN (1, 2, 3, 4, 5, 6, 99)
        ),
    store_and_fwd         BOOLEAN,
    pu_location_id        INTEGER NOT NULL
        REFERENCES hw3_taxi_zones (location_id),
    do_location_id        INTEGER NOT NULL
        REFERENCES hw3_taxi_zones (location_id),
    payment_type          SMALLINT NOT NULL
        CONSTRAINT chk_trips_payment_type CHECK (payment_type BETWEEN 0 AND 6),
    fare_amount           NUMERIC(10, 2) NOT NULL,
    extra                 NUMERIC(10, 2),
    mta_tax               NUMERIC(10, 2),
    tip_amount            NUMERIC(10, 2),
    tolls_amount          NUMERIC(10, 2),
    improvement_surcharge NUMERIC(10, 2),
    total_amount          NUMERIC(10, 2) NOT NULL,
    congestion_surcharge  NUMERIC(10, 2),
    airport_fee           NUMERIC(10, 2),
    cbd_congestion_fee    NUMERIC(10, 2),
    CONSTRAINT chk_trips_time_order CHECK (dropoff_at >= pickup_at)
);
''')

### 2.3. Очищення staging → final

Від прикладу із завдання відрізняються три речі. `store_and_fwd` нормалізується в `BOOLEAN`. Додано перевірку місяця: файл за жовтень має містити лише поїздки, що почалися в жовтні за місцевим часом, а межі задано напіввідкритим інтервалом. `pickup_local` рахується окремо від `pickup_at`, щоб порівняння не залежало від пояса сесії.

In [10]:
run('''
INSERT INTO hw3_taxi_zones (location_id, borough, zone, service_zone)
SELECT
    NULLIF(TRIM(location_id_raw), '')::INTEGER,
    CASE WHEN TRIM(borough_raw) IN ('', 'Unknown', 'N/A') THEN NULL ELSE TRIM(borough_raw) END,
    CASE WHEN TRIM(zone_raw) IN ('', 'Unknown', 'N/A') THEN NULL ELSE TRIM(zone_raw) END,
    CASE WHEN TRIM(service_zone_raw) IN ('', 'Unknown', 'N/A') THEN NULL ELSE TRIM(service_zone_raw) END
FROM hw3_taxi_zones_staging;
''')

run('''
WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::SMALLINT AS vendor_id,
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP AS pickup_local,
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP
            AT TIME ZONE 'America/New_York' AS pickup_at,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP
            AT TIME ZONE 'America/New_York' AS dropoff_at,
        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT AS passenger_count,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2) AS trip_distance,
        NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC::SMALLINT AS ratecode_id,
        CASE UPPER(TRIM(store_and_fwd_flag_raw))
            WHEN 'Y' THEN TRUE
            WHEN 'N' THEN FALSE
        END AS store_and_fwd,
        NULLIF(TRIM(pu_location_id_raw), '')::INTEGER AS pu_location_id,
        NULLIF(TRIM(do_location_id_raw), '')::INTEGER AS do_location_id,
        NULLIF(TRIM(payment_type_raw), '')::SMALLINT AS payment_type,
        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2) AS fare_amount,
        NULLIF(TRIM(extra_raw), '')::NUMERIC(10, 2) AS extra,
        NULLIF(TRIM(mta_tax_raw), '')::NUMERIC(10, 2) AS mta_tax,
        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2) AS tip_amount,
        NULLIF(TRIM(tolls_amount_raw), '')::NUMERIC(10, 2) AS tolls_amount,
        NULLIF(TRIM(improvement_surcharge_raw), '')::NUMERIC(10, 2) AS improvement_surcharge,
        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2) AS total_amount,
        NULLIF(TRIM(congestion_surcharge_raw), '')::NUMERIC(10, 2) AS congestion_surcharge,
        NULLIF(TRIM(airport_fee_raw), '')::NUMERIC(10, 2) AS airport_fee,
        NULLIF(TRIM(cbd_congestion_fee_raw), '')::NUMERIC(10, 2) AS cbd_congestion_fee
    FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
    vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
    ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, extra, mta_tax, tip_amount, tolls_amount,
    improvement_surcharge, total_amount, congestion_surcharge,
    airport_fee, cbd_congestion_fee
)
SELECT
    vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
    ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, extra, mta_tax, tip_amount, tolls_amount,
    improvement_surcharge, total_amount, congestion_surcharge,
    airport_fee, cbd_congestion_fee
FROM parsed
WHERE pickup_at IS NOT NULL
  AND dropoff_at IS NOT NULL
  AND dropoff_at >= pickup_at
  AND trip_distance >= 0
  AND pickup_local >= TIMESTAMP '2025-10-01 00:00:00'
  AND pickup_local <  TIMESTAMP '2025-11-01 00:00:00'
  AND (
        passenger_count IS NULL
        OR passenger_count >= 0
      );
''')

q('''
SELECT
    trip_id, vendor_id, pickup_at, dropoff_at, pickup_date,
    passenger_count, trip_distance, store_and_fwd,
    pu_location_id, do_location_id, payment_type, fare_amount, total_amount
FROM hw3_taxi_trips
ORDER BY pickup_at, trip_id
LIMIT 5
''')

,trip_id,vendor_id,pickup_at,dropoff_at,pickup_date,passenger_count,trip_distance,store_and_fwd,pu_location_id,do_location_id,payment_type,fare_amount,total_amount
0,77587,2,2025-10-01 00:00:07-04:00,2025-10-01 00:12:39-04:00,2025-10-01,NaN,1.87,None,161,141,0,14.07,18.82
1,12,7,2025-10-01 00:03:37-04:00,2025-10-01 00:03:37-04:00,2025-10-01,1.00,1.74,False,158,231,1,12.10,21.42
2,77590,2,2025-10-01 00:03:50-04:00,2025-10-01 00:16:36-04:00,2025-10-01,NaN,2.39,None,142,263,0,11.06,15.06
3,14,1,2025-10-01 00:04:51-04:00,2025-10-01 00:12:26-04:00,2025-10-01,1.00,1.10,False,249,79,1,7.90,17.75
4,11,2,2025-10-01 00:05:23-04:00,2025-10-01 00:12:26-04:00,2025-10-01,1.00,1.98,False,161,263,3,10.70,16.45


## Завдання 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [11]:
q('''
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_staging
UNION ALL
SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_trips
UNION ALL
SELECT 'zones_staging' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_zones_staging
UNION ALL
SELECT 'zones' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_zones
ORDER BY layer
''')

,layer,n_rows
0,clean,100000
1,staging,100000
2,zones,265
3,zones_staging,265


Перевіряємо кожне правило фільтра окремо, а також контрольну суму: кількість рядків могла б збігтися навіть за помилки перетворення.

In [12]:
q('''
WITH parsed AS (
    SELECT
        pickup_raw::TIMESTAMP AS pickup_local,
        dropoff_raw::TIMESTAMP AS dropoff_local,
        trip_distance_raw::NUMERIC AS trip_distance,
        passenger_count_raw::NUMERIC AS passenger_count
    FROM hw3_taxi_staging
)
SELECT
    COUNT(*) FILTER (WHERE pickup_local IS NULL OR dropoff_local IS NULL) AS rule_null_timestamp,
    COUNT(*) FILTER (WHERE dropoff_local < pickup_local) AS rule_dropoff_before_pickup,
    COUNT(*) FILTER (WHERE trip_distance < 0) AS rule_negative_distance,
    COUNT(*) FILTER (WHERE passenger_count < 0) AS rule_negative_passengers,
    COUNT(*) FILTER (
        WHERE pickup_local < TIMESTAMP '2025-10-01 00:00:00'
           OR pickup_local >= TIMESTAMP '2025-11-01 00:00:00'
    ) AS rule_outside_month
FROM parsed
''')

,rule_null_timestamp,rule_dropoff_before_pickup,rule_negative_distance,rule_negative_passengers,rule_outside_month
0,0,0,0,0,0


In [13]:
q('''
SELECT
    'staging' AS layer,
    ROUND(SUM(total_amount_raw::NUMERIC), 2) AS sum_total_amount,
    ROUND(SUM(trip_distance_raw::NUMERIC), 2) AS sum_trip_distance
FROM hw3_taxi_staging
UNION ALL
SELECT
    'clean' AS layer,
    SUM(total_amount),
    SUM(trip_distance)
FROM hw3_taxi_trips
ORDER BY layer
''')

,layer,sum_total_amount,sum_trip_distance
0,clean,2685488.50,548880.71
1,staging,2685488.50,548880.71


**Висновок.** У staging і clean однаково по 100 000 рядків, а в довіднику зон по 265. Правила фільтра задано в 2.3: відкидаються рядки без часу, з `dropoff_at < pickup_at`, із від'ємною відстанню чи кількістю пасажирів, а також із подачею поза інтервалом `[2025-10-01, 2025-11-01)`. У цій вибірці жодне правило не спрацювало: усі лічильники дорівнюють нулю. Це очікувано, бо таких рядків у всьому файлі лише 15, а за частки 2,26% на вибірку припадає менше одного.

Контрольні суми також збігаються: `total_amount` 2 685 488,50 і `trip_distance` 548 880,71 в обох шарах. Отже, під час приведення типів значення не втрачено.

### 3.2. Перевірка дублікатів

Гарантованого природного ключа у файлі немає, тому обидві перевірки виявляють лише точні або потенційні дублікати.

- Ключ A: `vendor_id`, час подачі й висадки, зони, `trip_distance`, `fare_amount`, `total_amount`. Це комбінація часу, локацій і тарифних атрибутів, тобто повторно записана поїздка.
- Ключ B: той самий ключ без сум. Він знаходить кілька записів про одну поїздку, зокрема пари нарахування й сторно.

In [14]:
q('''
SELECT
    'A: trip + amounts' AS business_key,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                    trip_distance, fare_amount, total_amount)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                               trip_distance, fare_amount, total_amount)) AS duplicate_rows
FROM hw3_taxi_trips
UNION ALL
SELECT
    'B: trip only' AS business_key,
    COUNT(*),
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                    trip_distance)),
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                               trip_distance))
FROM hw3_taxi_trips
ORDER BY business_key
''')

,business_key,total_rows,distinct_business_keys,duplicate_rows
0,A: trip + amounts,100000,100000,0
1,B: trip only,100000,99970,30


**Висновок.** За ключем A дублікатів немає: 0 зі 100 000. За ключем B знайдено 30 зайвих рядків, це потенційні дублікати, а не помилка. У всьому файлі таких груп 61 489, і майже всі є парами нарахування й сторно. Випадкова вибірка зберігає пару лише тоді, коли в неї потрапили обидва рядки, з імовірністю приблизно 0,0226². Очікується близько 31 пари, що відповідає результату.

Тому вибірка занижує цю проблему, а більшість сторно в ній залишилися без свого нарахування. Для ML це важливо: таку пару слід згортати до вибірки, на рівні всього файлу.

### 3.3. Null-rate і спеціальні маркери

Спочатку частка `NULL` у clean-таблиці. Результат транспоновано, щоб його було зручно читати.

In [15]:
q('''
SELECT
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
    ROUND(COUNT(*) FILTER (WHERE ratecode_id IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_ratecode_id,
    ROUND(COUNT(*) FILTER (WHERE store_and_fwd IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
    ROUND(COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_congestion_surcharge,
    ROUND(COUNT(*) FILTER (WHERE airport_fee IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_airport_fee,
    ROUND(COUNT(*) FILTER (WHERE tip_amount IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_tip_amount,
    ROUND(COUNT(*) FILTER (WHERE cbd_congestion_fee IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_cbd_congestion_fee,
    COUNT(*) AS total_rows
FROM hw3_taxi_trips
''').T.rename(columns={0: 'value'})

,value
pct_null_passenger_count,22.56
pct_null_ratecode_id,22.56
pct_null_store_and_fwd,22.56
pct_null_congestion_surcharge,22.56
pct_null_airport_fee,22.56
pct_null_tip_amount,0.00
pct_null_cbd_congestion_fee,0.00
total_rows,100000.00


Чи пов'язані ці пропуски між собою? Перевіряємо, у яких рядках вони трапляються.

In [16]:
q('''
SELECT
    (payment_type = 0) AS is_payment_type_0,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS null_passenger_count,
    COUNT(*) FILTER (WHERE ratecode_id IS NULL) AS null_ratecode_id,
    COUNT(*) FILTER (WHERE store_and_fwd IS NULL) AS null_store_and_fwd,
    COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) AS null_congestion_surcharge,
    COUNT(*) FILTER (WHERE airport_fee IS NULL) AS null_airport_fee
FROM hw3_taxi_trips
GROUP BY (payment_type = 0)
ORDER BY is_payment_type_0
''')

,is_payment_type_0,n_rows,null_passenger_count,null_ratecode_id,null_store_and_fwd,null_congestion_surcharge,null_airport_fee
0,False,77438,0,0,0,0,0
1,True,22562,22562,22562,22562,22562,22562


Далі сирі дані. Для всіх 20 колонок staging рахуємо `NULL`, порожні рядки, текстові маркери пропуску та значення, що не відповідають очікуваному формату. У режимі `FORMAT csv` порожнє поле без лапок `COPY` записує як `NULL`, тому порожні клітинки файлу потрапляють у колонку `n_sql_null`, а `n_empty_string` рахує лише рядки з пробілів або порожній рядок у лапках.

In [17]:
raw_markers = q('''
SELECT
    u.column_name,
    COUNT(*) FILTER (WHERE u.raw_value IS NULL) AS n_sql_null,
    COUNT(*) FILTER (WHERE TRIM(u.raw_value) = '') AS n_empty_string,
    COUNT(*) FILTER (
        WHERE LOWER(TRIM(u.raw_value)) IN ('?', 'unknown', 'n/a', 'na', 'null', 'none', 'nan')
    ) AS n_text_marker,
    COUNT(*) FILTER (WHERE TRIM(u.raw_value) !~ u.expected_format) AS n_bad_format
FROM hw3_taxi_staging AS s
CROSS JOIN LATERAL (
    VALUES
        ('vendor_id',             s.vendor_id_raw,             '^[0-9]+$'),
        ('pickup',                s.pickup_raw,                '^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$'),
        ('dropoff',               s.dropoff_raw,               '^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$'),
        ('passenger_count',       s.passenger_count_raw,       '^[0-9]+([.]0)?$'),
        ('trip_distance',         s.trip_distance_raw,         '^-?[0-9]+([.][0-9]+)?$'),
        ('ratecode_id',           s.ratecode_id_raw,           '^[0-9]+([.]0)?$'),
        ('store_and_fwd_flag',    s.store_and_fwd_flag_raw,    '^[YN]$'),
        ('pu_location_id',        s.pu_location_id_raw,        '^[0-9]+$'),
        ('do_location_id',        s.do_location_id_raw,        '^[0-9]+$'),
        ('payment_type',          s.payment_type_raw,          '^[0-9]+$'),
        ('fare_amount',           s.fare_amount_raw,           '^-?[0-9]+([.][0-9]+)?$'),
        ('extra',                 s.extra_raw,                 '^-?[0-9]+([.][0-9]+)?$'),
        ('mta_tax',               s.mta_tax_raw,               '^-?[0-9]+([.][0-9]+)?$'),
        ('tip_amount',            s.tip_amount_raw,            '^-?[0-9]+([.][0-9]+)?$'),
        ('tolls_amount',          s.tolls_amount_raw,          '^-?[0-9]+([.][0-9]+)?$'),
        ('improvement_surcharge', s.improvement_surcharge_raw, '^-?[0-9]+([.][0-9]+)?$'),
        ('total_amount',          s.total_amount_raw,          '^-?[0-9]+([.][0-9]+)?$'),
        ('congestion_surcharge',  s.congestion_surcharge_raw,  '^-?[0-9]+([.][0-9]+)?$'),
        ('airport_fee',           s.airport_fee_raw,           '^-?[0-9]+([.][0-9]+)?$'),
        ('cbd_congestion_fee',    s.cbd_congestion_fee_raw,    '^-?[0-9]+([.][0-9]+)?$')
) AS u(column_name, raw_value, expected_format)
GROUP BY u.column_name
ORDER BY n_sql_null DESC, u.column_name
''')
raw_markers

,column_name,n_sql_null,n_empty_string,n_text_marker,n_bad_format
0,airport_fee,22562,0,0,0
1,congestion_surcharge,22562,0,0,0
2,passenger_count,22562,0,0,0
3,ratecode_id,22562,0,0,0
4,store_and_fwd_flag,22562,0,0,0
5,cbd_congestion_fee,0,0,0,0
6,do_location_id,0,0,0,0
7,dropoff,0,0,0,0
8,extra,0,0,0,0
9,fare_amount,0,0,0,0


Текстових маркерів у числовому файлі може не бути, але в ньому є доменні маркери: це звичайні значення, які означають пропуск або особливий випадок. Перевіряємо їх окремо, а також маркери в довіднику зон.

In [18]:
q('''
SELECT
    COUNT(*) FILTER (WHERE passenger_count = 0) AS n_passenger_count_0,
    COUNT(*) FILTER (WHERE ratecode_id = 99) AS n_ratecode_99,
    COUNT(*) FILTER (WHERE payment_type = 0) AS n_payment_type_0,
    COUNT(*) FILTER (WHERE payment_type = 5) AS n_payment_type_5,
    COUNT(*) FILTER (WHERE pu_location_id IN (264, 265)) AS n_pickup_zone_264_265,
    COUNT(*) FILTER (WHERE do_location_id IN (264, 265)) AS n_dropoff_zone_264_265
FROM hw3_taxi_trips
''').T.rename(columns={0: 'n_rows'})

,n_rows
n_passenger_count_0,516
n_ratecode_99,2390
n_payment_type_0,22562
n_payment_type_5,0
n_pickup_zone_264_265,207
n_dropoff_zone_264_265,670


In [19]:
q('''
SELECT
    location_id_raw,
    borough_raw,
    zone_raw,
    service_zone_raw
FROM hw3_taxi_zones_staging
WHERE TRIM(borough_raw) IN ('Unknown', 'N/A')
   OR TRIM(zone_raw) IN ('Unknown', 'N/A')
   OR TRIM(service_zone_raw) IN ('Unknown', 'N/A')
ORDER BY location_id_raw::INTEGER
''')

,location_id_raw,borough_raw,zone_raw,service_zone_raw
0,264,Unknown,N/A,N/A
1,265,N/A,Outside of NYC,N/A


**Висновок.**

- П'ять колонок мають однаковий null-rate 22,56%, і це ті самі 22 562 рядки: усі вони мають `payment_type = 0`, а поза цим класом пропусків немає. Отже, пропуски не випадкові: вони описують окремий тип запису.
- У сирих даних текстових маркерів, порожніх рядків і значень у неочікуваному форматі немає: усі лічильники нульові. Формат файлу чистий, бо він згенерований із типізованого Parquet.
- Доменні маркери є: `passenger_count = 0` у 516 рядках, `ratecode_id = 99` у 2 390 рядках, зони 264 і 265 у 207 подачах і 670 висадках. Код `payment_type = 5` не трапився. У довіднику `Unknown` і `N/A` є лише в зонах 264 і 265, вони очищені до `NULL`, а назва `Outside of NYC` збережена.
- Нульовий null-rate не гарантує якості. `tip_amount` не має пропусків, але готівкові чайові в ньому не фіксуються. `dropoff_at` теж заповнений у всіх рядках, проте в 4.3 видно, що для постачальника 7 він просто дорівнює `pickup_at`.

## Завдання 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки: `payment_type`

In [20]:
q('''
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'Flex Fare'
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
    END AS payment_label,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10
''')

,payment_type,payment_label,n_rows,pct_rows
0,1,Credit card,65548,65.55
1,0,Flex Fare,22562,22.56
2,2,Cash,9130,9.13
3,4,Dispute,2164,2.16
4,3,No charge,596,0.60


**Висновок.** Домінує оплата карткою (65,55%). Другий за розміром клас Flex Fare (22,56%) більший за готівку (9,13%), тому його не можна просто відкинути без втрати п'ятої частини поїздок. `Dispute` (2,16%) і `No charge` (0,60%) є рідкісними категоріями, а коди 5 і 6 у вибірці відсутні. Такий дисбаланс означає, що середні показники без розбивки за оплатою описують насамперед картки.

### 4.2. Базова статистика числових колонок

In [21]:
q('''
SELECT
    'trip_distance' AS metric,
    COUNT(trip_distance) AS n_non_null,
    MIN(trip_distance) AS min_value,
    MAX(trip_distance) AS max_value,
    ROUND(AVG(trip_distance)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(trip_distance)::NUMERIC, 2) AS sd_value,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_distance) AS median_value,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance) AS p99_value
FROM hw3_taxi_trips
UNION ALL
SELECT
    'total_amount',
    COUNT(total_amount),
    MIN(total_amount),
    MAX(total_amount),
    ROUND(AVG(total_amount)::NUMERIC, 2),
    ROUND(STDDEV_SAMP(total_amount)::NUMERIC, 2),
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_amount),
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY total_amount)
FROM hw3_taxi_trips
''')

,metric,n_non_null,min_value,max_value,avg_value,sd_value,median_value,p99_value
0,trip_distance,100000,0.00,133670.99,5.49,440.01,1.89,19.60
1,total_amount,100000,-801.00,702.37,26.85,24.30,21.42,105.60


**Висновок.** Для `trip_distance` середнє 5,49 милі майже втричі більше за медіану 1,89, а стандартне відхилення 440,01 значно перевищує навіть 99-й перцентиль 19,60. Так виглядає вплив кількох помилкових значень: максимум становить 133 670,99 милі. `total_amount` стабільніший (середнє 26,85, медіана 21,42), але діапазон від -801,00 до 702,37 показує сторно й договірні тарифи. Для цих колонок медіана й перцентилі надійніші за середнє та σ.

### 4.3. Outlier-check

Основна перевірка доменна, бо розподіли дуже скошені. Порогове значення `total_amount > 10000` із прикладу тут не спрацює: максимум значно менший. Тому пороги взято з предметної області: поїздка понад 100 миль виходить далеко за межі міста, понад 6 годин роботи таксометра для однієї поїздки є нетиповими.

In [22]:
q('''
SELECT
    COUNT(*) FILTER (WHERE fare_amount < 0 AND total_amount < 0) AS n_negative_fare_and_total,
    COUNT(*) FILTER (WHERE fare_amount < 0 AND total_amount >= 0) AS n_negative_fare_positive_total,
    COUNT(*) FILTER (
        WHERE fare_amount < 0 AND total_amount >= 0 AND payment_type = 0
    ) AS n_negative_fare_positive_total_flex,
    COUNT(*) FILTER (WHERE tip_amount < 0) AS n_negative_tip,
    COUNT(*) FILTER (WHERE trip_distance = 0) AS n_zero_distance,
    COUNT(*) FILTER (WHERE trip_distance = 0 AND ratecode_id = 5) AS n_zero_distance_negotiated,
    COUNT(*) FILTER (WHERE trip_distance > 100) AS n_distance_over_100_miles,
    COUNT(*) FILTER (WHERE dropoff_at = pickup_at) AS n_zero_duration,
    COUNT(*) FILTER (WHERE dropoff_at = pickup_at AND vendor_id = 7) AS n_zero_duration_vendor_7,
    COUNT(*) FILTER (WHERE dropoff_at - pickup_at > INTERVAL '6 hours') AS n_duration_over_6_hours,
    COUNT(*) FILTER (WHERE passenger_count > 6) AS n_passengers_over_6,
    COUNT(*) FILTER (WHERE total_amount > 500) AS n_total_over_500
FROM hw3_taxi_trips
''').T.rename(columns={0: 'n_rows'})

,n_rows
n_negative_fare_and_total,2508
n_negative_fare_positive_total,4869
n_negative_fare_positive_total_flex,4869
n_negative_tip,4
n_zero_distance,2850
n_zero_distance_negotiated,347
n_distance_over_100_miles,6
n_zero_duration,1564
n_zero_duration_vendor_7,1548
n_duration_over_6_hours,29


Від'ємний тариф за додатного підсумку трапляється лише у Flex Fare, тому порівнюємо рядки цього класу з від'ємним тарифом і решту його рядків.

In [23]:
q('''
SELECT
    (fare_amount < 0) AS is_negative_fare,
    COUNT(*) AS n_rows,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_distance) AS median_distance,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_amount) AS median_total
FROM hw3_taxi_trips
WHERE payment_type = 0
GROUP BY (fare_amount < 0)
ORDER BY is_negative_fare
''')

,is_negative_fare,n_rows,median_distance,median_total
0,False,16731,2.70,22.12
1,True,5831,2.63,3.00


Конкретні рядки з найбільшими значеннями відстані, разом із часом і сумою, щоб оцінити їхню правдоподібність.

In [24]:
q('''
SELECT
    trip_id,
    vendor_id,
    payment_type,
    pickup_at,
    dropoff_at,
    pu_location_id,
    do_location_id,
    trip_distance,
    fare_amount,
    total_amount
FROM hw3_taxi_trips
WHERE trip_distance > 100
ORDER BY trip_distance DESC
LIMIT 20
''')

,trip_id,vendor_id,payment_type,pickup_at,dropoff_at,pu_location_id,do_location_id,trip_distance,fare_amount,total_amount
0,86308,2,0,2025-10-13 07:29:00-04:00,2025-10-13 07:41:00-04:00,236,238,133670.99,15.17,19.17
1,81647,2,0,2025-10-07 08:49:00-04:00,2025-10-07 09:08:00-04:00,148,13,24995.69,22.04,28.13
2,98140,2,0,2025-10-30 12:45:00-04:00,2025-10-30 12:54:00-04:00,238,166,23921.86,3.78,5.28
3,83121,2,0,2025-10-09 14:29:00-04:00,2025-10-09 14:48:00-04:00,107,233,17181.09,11.14,15.89
4,7224,7,1,2025-10-03 22:46:01-04:00,2025-10-03 22:46:01-04:00,138,265,418.72,66.00,75.25
5,77197,2,1,2025-10-31 21:29:15-04:00,2025-11-01 01:30:23-04:00,246,265,229.51,300.00,378.97


Рядки з найбільшими сумами та зразок від'ємних сум поза `payment_type = 0`.

In [25]:
q('''
(
    SELECT
        'largest_total' AS check_name,
        trip_id, vendor_id, payment_type, ratecode_id,
        pu_location_id, do_location_id, trip_distance,
        fare_amount, tip_amount, total_amount
    FROM hw3_taxi_trips
    ORDER BY total_amount DESC, trip_id
    LIMIT 5
)
UNION ALL
(
    SELECT
        'most_negative_total' AS check_name,
        trip_id, vendor_id, payment_type, ratecode_id,
        pu_location_id, do_location_id, trip_distance,
        fare_amount, tip_amount, total_amount
    FROM hw3_taxi_trips
    WHERE payment_type <> 0
    ORDER BY total_amount, trip_id
    LIMIT 5
)
''')

,check_name,trip_id,vendor_id,payment_type,ratecode_id,pu_location_id,do_location_id,trip_distance,fare_amount,tip_amount,total_amount
0,largest_total,9521,2,2,5,230,230,0.00,670.00,0.00,702.37
1,largest_total,28881,2,4,5,28,28,0.00,480.00,0.00,481.00
2,largest_total,69683,2,1,5,265,265,0.00,422.65,50.00,473.65
3,largest_total,28127,2,1,5,265,265,0.00,392.00,50.00,443.00
4,largest_total,28894,2,1,5,238,238,0.00,400.00,0.00,403.50
5,most_negative_total,29440,2,3,5,216,216,0.00,-800.00,0.00,-801.00
6,most_negative_total,62438,2,3,5,132,132,0.00,-500.00,0.00,-501.00
7,most_negative_total,7362,2,3,4,132,265,71.93,-488.80,0.00,-493.05
8,most_negative_total,74260,2,4,5,132,132,0.00,-450.00,0.00,-451.00
9,most_negative_total,15153,2,4,5,138,138,0.00,-300.00,0.00,-307.75


Додатково правило 3σ для `trip_distance`. Воно показує, чому для цієї колонки статистичний варіант гірший за доменний.

In [26]:
q('''
WITH stats AS (
    SELECT
        AVG(trip_distance) AS mean_value,
        STDDEV_SAMP(trip_distance) AS sd_value
    FROM hw3_taxi_trips
    WHERE trip_distance IS NOT NULL
)
SELECT
    t.trip_id,
    t.vendor_id,
    t.payment_type,
    t.trip_distance,
    t.total_amount,
    ROUND(s.mean_value, 2) AS mean_value,
    ROUND(s.sd_value, 2) AS sd_value,
    ROUND(
        ((t.trip_distance - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_taxi_trips AS t
CROSS JOIN stats AS s
WHERE t.trip_distance IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.trip_distance - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.trip_distance - s.mean_value) DESC
LIMIT 20
''')

,trip_id,vendor_id,payment_type,trip_distance,total_amount,mean_value,sd_value,z_score
0,86308,2,0,133670.99,19.17,5.49,440.01,303.78
1,81647,2,0,24995.69,28.13,5.49,440.01,56.79
2,98140,2,0,23921.86,5.28,5.49,440.01,54.35
3,83121,2,0,17181.09,15.89,5.49,440.01,39.03


**Висновок.** Від'ємний тариф за додатного підсумку є в 4 869 рядках, і всі вони належать до Flex Fare. Рядки Flex Fare з від'ємним тарифом (5 831) мають майже таку саму медіанну відстань, як інші рядки цього класу (2,63 проти 2,70 милі), але медіанний підсумок лише 3,00 проти 22,12. Тому в цих рядках `fare_amount` і `total_amount`, схоже, не описують повну вартість поїздки. Це припущення за даними, його треба уточнити в джерела.

Правило 3σ тут працює погано. Поріг `mean + 3σ` дорівнює приблизно 1 325 милям, бо σ роздута цими самими помилками. Тому воно знаходить лише 4 рядки й не бачить поїздку на 418,72 милі, яка явно підозріла. Доменні правила знаходять її, тому вони є основними.

| Аномалія | Рядків | Оцінка | Дія |
|---|---:|---|---|
| Від'ємні `fare_amount` і `total_amount` | 2 508 | сторно в `Dispute`, `Cash`, `No charge`, а також від'ємні записи Flex Fare | залишити в clean, додати прапорець, виключити з training data |
| Від'ємний тариф за додатного підсумку | 4 869 | особливість Flex Fare | передати на перевірку джерела, не використовувати `fare_amount` цього класу як target |
| Від'ємні чайові | 4 | сторно | додати прапорець |
| Нульова відстань | 2 850 | різнорідна група: договірний тариф (`ratecode_id = 5`) мають лише 347 рядків, зокрема всі з топ-5 чеків | залишити як валідний випадок, додати прапорець, виключити з моделей вартості за милю |
| Відстань понад 17 000 миль | 4 | усі Flex Fare постачальника 2, 9–19 хв, помилка одиниць або лічильника | виключити з training data, передати на перевірку джерела |
| 229,51 милі, 4 год, тариф 300,00 | 1 | узгоджені відстань, час і сума | залишити як валідний рідкісний випадок |
| 418,72 милі, нульова тривалість, тариф 66,00 | 1 | відстань не відповідає сумі | виключити з training data |
| Нульова тривалість | 1 564 | 1 548 із них є всіма рядками постачальника 7 у вибірці | передати на перевірку джерела, для цього постачальника не рахувати тривалість |
| Тривалість понад 6 год | 29 | імовірно, таксометр не вимкнули вчасно | додати прапорець, виключити з моделей тривалості |
| `total_amount` понад 500 | 1 | 702,37, договірний тариф, нульова відстань | залишити як валідний рідкісний випадок |

`passenger_count > 6` не трапився. Значення не видалено лише через їхню величину: усі ці рядки залишено в clean, а рішення стосуються прапорців і відбору для навчання.

## Завдання 5. DQL/EDA-запити

Виконано 12 запитів. Перші чотири відповідають на орієнтовні запитання для NYC Taxi. `JOIN` із `hw3_taxi_zones` виконується за первинним ключем, тому кількість рядків не зростає. Години рахуються за місцевим часом через `AT TIME ZONE`. Покриття конструкцій подано в таблиці наприкінці розділу.

### Запит 1
**Business-question:** Які `payment_type` найпоширеніші та скільки в кожному з них припадає на середній чек і чайові?

In [27]:
q('''
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'Flex Fare'
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
    END AS payment_label,
    COUNT(*) AS n_trips,
    ROUND(AVG(total_amount), 2) AS avg_total,
    ROUND(
        SUM(tip_amount) * 100.0 / NULLIF(SUM(fare_amount), 0),
        2
    ) AS tip_pct_of_fare
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_trips DESC, payment_type
''')

,payment_type,payment_label,n_trips,avg_total,tip_pct_of_fare
0,1,Credit card,65548,31.11,21.42
1,0,Flex Fare,22562,18.69,2.36
2,2,Cash,9130,23.91,0.00
3,4,Dispute,2164,2.35,-0.38
4,3,No charge,596,2.12,0.00


**Interpretation:** Найчастіше платять карткою (65 548 поїздок), далі йдуть Flex Fare (22 562) і готівка (9 130). Чайові становлять 21,42% від тарифу в картках, 2,36% у Flex Fare і 0,00% у готівці, бо готівкові чайові не записуються, тому порівнювати щедрість пасажирів за цією колонкою не можна. Середній чек `Dispute` (2,35) і `No charge` (2,12) близький до нуля, бо нарахування й сторно майже компенсують одне одного.

### Запит 2
**Business-question:** У які години доби за місцевим часом найбільше поїздок?

In [28]:
q('''
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INTEGER AS pickup_hour,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_trips
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY n_trips DESC, pickup_hour
LIMIT 5
''')

,pickup_hour,n_trips,pct_trips
0,18,7200,7.20
1,17,6545,6.55
2,19,6447,6.45
3,21,5959,5.96
4,16,5950,5.95


**Interpretation:** Найбільше поїздок о 18:00 (7 200, або 7,20%), далі 17:00 і 19:00. Усі п'ять найзавантаженіших годин припадають на вечір з 16:00 до 21:59, разом 32,10% поїздок, а 20:00 до п'ятірки не входить. Це аргумент на користь години подачі як ознаки в моделях попиту. Обчислення за UTC зсунуло б пік на чотири години, тому година береться за місцевим часом.

### Запит 3
**Business-question:** Яка частка поїздок має `NULL` у `passenger_count` і чи залежить вона від постачальника?

In [29]:
q('''
SELECT
    COALESCE(vendor_id::TEXT, 'all') AS vendor,
    COUNT(*) AS n_trips,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS n_null_passenger_count,
    ROUND(
        COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*),
        2
    ) AS pct_null_passenger_count
FROM hw3_taxi_trips
GROUP BY ROLLUP (vendor_id)
ORDER BY vendor_id NULLS LAST
''')

,vendor,n_trips,n_null_passenger_count,pct_null_passenger_count
0,1,19783,3091,15.62
1,2,78592,19394,24.68
2,6,77,77,100.00
3,7,1548,0,0.00
4,all,100000,22562,22.56


**Interpretation:** `passenger_count` порожній у 22,56% поїздок. Частка залежить від постачальника: 15,62% у постачальника 1, 24,68% у постачальника 2, 100,00% у постачальника 6 і 0,00% у постачальника 7. Згідно з 3.3 ці рядки точно збігаються з `payment_type = 0`, тому важливі не самі постачальники, а частка Flex Fare в них. Заповнювати пропуск середнім не варто, бо відсутність значення сама несе інформацію про тип запису.

### Запит 4
**Business-question:** Які поїздки мають від'ємні суми або нульову відстань, якщо не враховувати вже відомий клас Flex Fare з від'ємним тарифом і додатним підсумком?

In [30]:
q('''
SELECT
    payment_type,
    COUNT(*) AS n_trips,
    COUNT(*) FILTER (WHERE total_amount < 0) AS n_negative_total,
    COUNT(*) FILTER (WHERE trip_distance = 0) AS n_zero_distance,
    COUNT(*) FILTER (WHERE trip_distance = 0 AND total_amount > 0) AS n_zero_distance_paid,
    COUNT(*) FILTER (WHERE trip_distance = 0 AND ratecode_id = 5) AS n_zero_distance_negotiated,
    ROUND(AVG(total_amount), 2) AS avg_total
FROM hw3_taxi_trips
WHERE (fare_amount < 0 OR total_amount < 0 OR trip_distance = 0)
  AND NOT (
        payment_type = 0
        AND fare_amount < 0
        AND total_amount >= 0
      )
GROUP BY payment_type
ORDER BY n_trips DESC, payment_type
''')

,payment_type,n_trips,n_negative_total,n_zero_distance,n_zero_distance_paid,n_zero_distance_negotiated,avg_total
0,0,2228,962,1373,1265,0,12.04
1,4,1076,990,136,83,14,-26.85
2,1,569,10,566,559,307,56.78
3,2,528,338,210,190,20,-10.20
4,3,298,211,129,87,6,-22.32


**Interpretation:** Поза відомим класом знайдено 4 699 рядків, найбільше у Flex Fare (2 228, з них 962 із від'ємним підсумком і 1 373 із нульовою відстанню). Від'ємні підсумки є також у `Dispute` (990), `Cash` (338) і `No charge` (211), а в картках таких лише 10. Картки мають 566 поїздок із нульовою відстанню, 559 з яких оплачені, а 307 мають договірний тариф; середній чек усіх 569 карткових рядків становить 56,78, тож більшість із них схожа на реальні поїздки без показань відстані, а не на пусті записи. Отже, правило «нульова відстань означає помилку» надто грубе, і такі рядки краще позначати, ніж видаляти.

### Запит 5
**Business-question:** Чим відрізняються поїздки з аеропортових зон за кількістю, відстанню та чеком?

Враховано лише картку й готівку: суми Flex Fare, `Dispute` і `No charge` часто від'ємні або неповні й спотворили б середній чек.

In [31]:
q('''
SELECT
    pu.zone AS pickup_zone,
    COUNT(*) AS n_trips,
    ROUND(AVG(t.trip_distance), 2) AS avg_distance,
    ROUND(AVG(t.total_amount), 2) AS avg_total
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu
    ON pu.location_id = t.pu_location_id
WHERE pu.zone ILIKE '%airport%'
  AND t.payment_type IN (1, 2)
GROUP BY pu.zone
ORDER BY n_trips DESC, pu.zone
''')

,pickup_zone,n_trips,avg_distance,avg_total
0,JFK Airport,3966,15.08,80.39
1,LaGuardia Airport,2602,9.66,71.30
2,Newark Airport,8,0.15,131.98


In [32]:
q('''
SELECT
    ROUND(AVG(t.trip_distance), 2) AS avg_distance_all_zones,
    ROUND(AVG(t.total_amount), 2) AS avg_total_all_zones,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips AS t
WHERE t.payment_type IN (1, 2)
''')

,avg_distance_all_zones,avg_total_all_zones,n_trips
0,3.44,30.23,74678


**Interpretation:** JFK Airport (3 966 поїздок, 15,08 милі, чек 80,39) і LaGuardia Airport (2 602, 9,66 милі, 71,30) разом дають близько 8,8% поїздок із відомою оплатою карткою чи готівкою. Чек із JFK у 2,7 раза більший за загальний середній (30,23). Newark Airport має лише 8 подач, тому середні 0,15 милі й 131,98 ненадійні. Для моделей вартості зона подачі є сильною ознакою.

### Запит 6
**Business-question:** Чи дорівнює `total_amount` сумі складових чека в кожного постачальника?

Перевіряємо дві формули: повну суму всіх складових і суму без трьох окремих зборів. Рядки `payment_type = 0` виключено, бо в них збори порожні. В інших рядках усі складові заповнені (див. 3.3), тому достатньо звичайного `<>`.

In [33]:
q('''
SELECT
    vendor_id,
    COUNT(*) AS n_trips,
    ROUND(
        COUNT(*) FILTER (
            WHERE total_amount <>
                  fare_amount + extra + mta_tax + tip_amount + tolls_amount
                  + improvement_surcharge + congestion_surcharge
                  + airport_fee + cbd_congestion_fee
        ) * 100.0 / COUNT(*),
        2
    ) AS pct_mismatch_all_components,
    ROUND(
        COUNT(*) FILTER (
            WHERE total_amount <>
                  fare_amount + extra + mta_tax + tip_amount + tolls_amount
                  + improvement_surcharge
        ) * 100.0 / COUNT(*),
        2
    ) AS pct_mismatch_without_fees,
    ROUND(AVG(extra), 2) AS avg_extra
FROM hw3_taxi_trips
WHERE payment_type <> 0
GROUP BY vendor_id
ORDER BY vendor_id
''')

,vendor_id,n_trips,pct_mismatch_all_components,pct_mismatch_without_fees,avg_extra
0,1,16692,83.54,0.00,3.44
1,2,59198,0.16,97.39,1.00
2,7,1548,46.77,98.97,0.00


**Interpretation:** Єдиної формули немає: для постачальника 2 повна сума складових збігається майже завжди (0,16% розбіжностей). Для постачальника 1 розбіжностей 83,54%, але без трьох зборів їх 0,00%, а середній `extra` становить 3,44 проти 1,00: його `extra` вже містить ці збори, і вони дублюються в окремих колонках. Постачальник 7 має `extra` 0,00 і 46,77% розбіжностей: частина доплат у підсумку є, але в складових не показана. Тому складові чека не можна порівнювати між постачальниками без нормалізації.

### Запит 7
**Business-question:** Які міжрайонні напрямки найпопулярніші, якщо зони без відомого району теж враховувати як окрему групу?

Після очищення зони 264 і 265 мають `borough IS NULL`. Звичайне `<>` мовчки відкинуло б такі рядки, а `IS DISTINCT FROM` їх залишає.

In [34]:
q('''
SELECT
    COALESCE(pu.borough, 'unknown') AS pickup_borough,
    COALESCE(dz.borough, 'unknown') AS dropoff_borough,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu
    ON pu.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz
    ON dz.location_id = t.do_location_id
WHERE pu.borough IS DISTINCT FROM dz.borough
GROUP BY pu.borough, dz.borough
ORDER BY n_trips DESC, pickup_borough, dropoff_borough
LIMIT 10
''')

,pickup_borough,dropoff_borough,n_trips
0,Queens,Manhattan,4892
1,Manhattan,Brooklyn,2664
2,Manhattan,Queens,2564
3,Queens,Brooklyn,1517
4,Brooklyn,Manhattan,1310
5,Manhattan,Bronx,686
6,Brooklyn,Queens,454
7,Bronx,Manhattan,369
8,Manhattan,unknown,295
9,Queens,unknown,255


In [35]:
q('''
SELECT
    COUNT(*) FILTER (WHERE pu.borough IS DISTINCT FROM dz.borough) AS n_is_distinct_from,
    COUNT(*) FILTER (WHERE pu.borough <> dz.borough) AS n_not_equal,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu
    ON pu.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz
    ON dz.location_id = t.do_location_id
''')

,n_is_distinct_from,n_not_equal,n_trips
0,15760,15099,100000


**Interpretation:** Найпопулярніший напрямок Queens → Manhattan (4 892 поїздки), що майже вдвічі більше за зворотний Manhattan → Queens (2 564). Далі Manhattan → Brooklyn (2 664). Усього міжрайонних поїздок 15 760, або 15,76% вибірки. Звичайне `<>` дало б 15 099, тобто втратило б 661 рядок із зонами без відомого району, серед яких поїздки за межі міста.

### Запит 8
**Business-question:** Які комбінації постачальника й способу оплати реально трапляються в даних?

In [36]:
q('''
SELECT DISTINCT
    vendor_id,
    payment_type
FROM hw3_taxi_trips
ORDER BY vendor_id, payment_type
''')

,vendor_id,payment_type
0,1,0
1,1,1
2,1,2
3,1,3
4,1,4
5,2,0
6,2,1
7,2,2
8,2,3
9,2,4


**Interpretation:** У даних є 15 комбінацій. Постачальники 1 і 2 мають коди 0–4, постачальник 6 лише код 0, а постачальник 7 коди 1–4 без Flex Fare. Коди 5 і 6 не трапляються. Отже, `vendor_id` частково визначає тип запису, тому як ознака моделі він може відображати правила запису даних, а не поведінку пасажирів.

### Запит 9
**Business-question:** Який день тижня найзавантаженіший, якщо порівнювати рівну кількість кожного дня?

Жовтень містить неоднакову кількість різних днів тижня, тому взято чотири повні тижні: 4–31 жовтня. `COUNT(DISTINCT pickup_date)` підтверджує, що кожен день тижня представлений чотирма датами.

In [37]:
q('''
SELECT
    EXTRACT(ISODOW FROM pickup_date)::INTEGER AS iso_day_of_week,
    TO_CHAR(pickup_date, 'FMDay') AS day_name,
    COUNT(DISTINCT pickup_date) AS n_dates,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*)::NUMERIC / COUNT(DISTINCT pickup_date), 1) AS avg_trips_per_date
FROM hw3_taxi_trips
WHERE pickup_date BETWEEN DATE '2025-10-04' AND DATE '2025-10-31'
GROUP BY iso_day_of_week, day_name
ORDER BY iso_day_of_week
''')

,iso_day_of_week,day_name,n_dates,n_trips,avg_trips_per_date
0,1,Monday,4,10670,2667.50
1,2,Tuesday,4,12261,3065.30
2,3,Wednesday,4,13377,3344.30
3,4,Thursday,4,14496,3624.00
4,5,Friday,4,14482,3620.50
5,6,Saturday,4,13687,3421.80
6,7,Sunday,4,11768,2942.00


**Interpretation:** Найзавантаженіші четвер (3 624,0 поїздки вибірки на дату) і п'ятниця (3 620,5), практично на одному рівні. Найнижчий понеділок (2 667,5), на 26% менше за четвер, за ним неділя (2 942,0). Субота (3 421,8) перевищує вівторок, тому попит не зводиться до поділу «будній день або вихідний». Для прогнозу варто використовувати саме день тижня.

### Запит 10
**Business-question:** Як частка чайових залежить від відстані поїздки в оплатах карткою?

Готівкові чайові не потрапляють до запису, тому вибрано лише `payment_type = 1`. `NULLIF` захищає від ділення на нульовий тариф, а такі рядки не беруть участі в медіані.

In [38]:
q('''
SELECT
    CASE
        WHEN trip_distance < 1 THEN '1) under 1 mi'
        WHEN trip_distance < 3 THEN '2) 1-3 mi'
        WHEN trip_distance < 10 THEN '3) 3-10 mi'
        ELSE '4) 10-100 mi'
    END AS distance_band,
    COUNT(*) AS n_trips,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(
        (PERCENTILE_CONT(0.5) WITHIN GROUP (
            ORDER BY tip_amount * 100.0 / NULLIF(fare_amount, 0)
        ))::NUMERIC,
        2
    ) AS median_tip_pct_of_fare,
    ROUND(
        COUNT(*) FILTER (WHERE tip_amount = 0) * 100.0 / COUNT(*),
        2
    ) AS pct_zero_tip
FROM hw3_taxi_trips
WHERE payment_type = 1
  AND trip_distance > 0
  AND trip_distance <= 100
  AND tip_amount IS NOT NULL
GROUP BY distance_band
ORDER BY distance_band
''')

,distance_band,n_trips,avg_tip,median_tip_pct_of_fare,pct_zero_tip
0,1) under 1 mi,14784,2.50,32.31,5.40
1,2) 1-3 mi,31550,3.41,26.85,5.01
2,3) 3-10 mi,12987,5.87,23.51,13.38
3,4) 10-100 mi,5659,11.41,21.36,20.64


**Interpretation:** Середні чайові зростають із 2,50 до 11,41 зі збільшенням відстані, але медіанна частка від тарифу падає з 32,31% до 21,36%. Отже, чайові зростають повільніше за тариф, а їхня абсолютна сума значною мірою повторює відстань. Частка нульових чайових зростає з 5,40% на коротких поїздках до 20,64% на довгих. Для моделі чайових цікавіші частка й факт наявності чайових, ніж абсолютна сума.

### Запит 11
**Business-question:** У які години робочого дня таксі рухаються найповільніше?

Без очищення цей показник хибний. Тому виключено постачальника 7 з нульовою тривалістю, а також нульові й надвеликі відстані й тривалості.

In [39]:
q('''
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INTEGER AS pickup_hour,
    COUNT(*) AS n_trips,
    ROUND(
        (PERCENTILE_CONT(0.5) WITHIN GROUP (
            ORDER BY trip_distance / (EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600)
        ))::NUMERIC,
        2
    ) AS median_speed_mph
FROM hw3_taxi_trips
WHERE vendor_id <> 7
  AND dropoff_at > pickup_at
  AND dropoff_at - pickup_at <= INTERVAL '6 hours'
  AND trip_distance BETWEEN 0.1 AND 100
  AND EXTRACT(ISODOW FROM pickup_date) BETWEEN 1 AND 5
GROUP BY pickup_hour
ORDER BY median_speed_mph, pickup_hour
LIMIT 5
''')

,pickup_hour,n_trips,median_speed_mph
0,11,3367,7.05
1,15,4095,7.06
2,14,3950,7.25
3,12,3456,7.27
4,13,3624,7.39


**Interpretation:** Найповільніші години буднів припадають на день, а не на вечірній пік попиту: 11:00 (7,05 милі/год), 15:00 (7,06), 14:00 (7,25), 12:00 (7,27) і 13:00 (7,39). Найбільша кількість поїздок із запиту 2 припадає на 18:00, тож максимум попиту й мінімум швидкості не збігаються. Для прогнозу тривалості це означає, що година потрібна як окрема ознака, а не лише через кількість поїздок. Результат дійсний лише для відфільтрованих рядків без постачальника 7.

### Запит 12
**Business-question:** Які зони подачі найбільші за кількістю поїздок і яка частка поїздок у них завершується в тій самій зоні?

In [40]:
q('''
SELECT
    t.pu_location_id,
    COALESCE(pu.zone, 'unknown') AS pickup_zone,
    COALESCE(pu.borough, 'unknown') AS pickup_borough,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_trips,
    ROUND(
        COUNT(*) FILTER (
            WHERE t.pu_location_id = t.do_location_id
        ) * 100.0 / COUNT(*),
        2
    ) AS pct_same_zone
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu
    ON pu.location_id = t.pu_location_id
GROUP BY t.pu_location_id, pu.zone, pu.borough
ORDER BY n_trips DESC, t.pu_location_id
LIMIT 10
''')

,pu_location_id,pickup_zone,pickup_borough,n_trips,pct_trips,pct_same_zone
0,237,Upper East Side South,Manhattan,4645,4.65,10.16
1,132,JFK Airport,Queens,4428,4.43,5.60
2,161,Midtown Center,Manhattan,4408,4.41,4.72
3,236,Upper East Side North,Manhattan,4002,4.00,9.87
4,142,Lincoln Square East,Manhattan,3051,3.05,4.59
5,162,Midtown East,Manhattan,3044,3.04,3.65
6,186,Penn Station/Madison Sq West,Manhattan,3003,3.00,2.10
7,230,Times Sq/Theatre District,Manhattan,2996,3.00,3.94
8,138,LaGuardia Airport,Queens,2733,2.73,2.49
9,170,Murray Hill,Manhattan,2639,2.64,3.11


**Interpretation:** Десять найбільших зон дають 34,95% поїздок: вісім у Manhattan, а також JFK Airport і LaGuardia Airport. Перше місце має Upper East Side South (4 645 поїздок). Найбільша частка поїздок у межах зони в Upper East Side South (10,16%) і Upper East Side North (9,87%), а найменша в Penn Station/Madison Sq West (2,10%). JFK тут має 4 428 поїздок, а в запиті 5 менше, бо там враховано лише картку й готівку.

### Покриття обов'язкових конструкцій

| Конструкція | Запити |
|---|---|
| Явний перелік колонок у `SELECT` | усі, зокрема 1, 5, 8, 12 |
| `WHERE` з `AND` / `OR` / `NOT` і дужками | 4 |
| `IN` або `BETWEEN` | `IN`: 5; `BETWEEN`: 9, 11 |
| `LIKE` або `ILIKE` | `ILIKE`: 5 |
| `IS NULL` або `IS NOT NULL` | `IS NULL`: 3; `IS NOT NULL`: 10 |
| `COALESCE` або `NULLIF` | `COALESCE`: 3, 7, 12; `NULLIF`: 1, 10 |
| `IS DISTINCT FROM` або `IS NOT DISTINCT FROM` | `IS DISTINCT FROM`: 7 |
| `DISTINCT` | `SELECT DISTINCT`: 8; `COUNT(DISTINCT ...)`: 9 |
| `ORDER BY` разом із `LIMIT` | 2, 7, 11, 12 |
| Щонайменше два запити з `GROUP BY` | 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12 |

## Завдання 6. Reflection

**Якість даних.** Головна проблема: усі рядки з `payment_type = 0`, приблизно п'ята частина вибірки, мають одразу п'ять порожніх колонок, а їхній `fare_amount` часто від'ємний за додатного підсумку. Текстових маркерів немає, але є доменні: `ratecode_id = 99`, `passenger_count = 0`, зони 264 і 265. Постачальник 7 записує час висадки рівним часу подачі, постачальник 1 дублює збори в `extra`, сторно утворюють пари рядків, а відстань інколи сягає десятків тисяч миль. Точних дублікатів немає.

**Features.** Використав би годину й день тижня подачі, `pu_location_id`, а також `do_location_id`, якщо маршрут відомий заздалегідь. `passenger_count` брав би лише разом з індикатором пропуску. Не використав би `trip_id`, бо це технічний ключ, і `vendor_id`, бо він більше відображає правила запису, ніж поїздку. `fare_amount` із Flex Fare не брав би навіть як target. Рядки сторно виключив би з навчання.

**Leakage.** Для прогнозу на момент подачі `dropoff_at`, фактична `trip_distance`, усі суми, `tip_amount`, `tolls_amount` і `payment_type` стають відомими лише після поїздки. Те саме стосується `ratecode_id`: за словником TLC це підсумковий код наприкінці поїздки. Індикатор пропуску в `passenger_count` дорівнює `payment_type = 0`, тому він допустимий, лише якщо Flex Fare відомий до подачі. Під час прогнозу чайових заборонений також `total_amount`, бо він їх містить.

**Pipeline.** Запропонував би data engineer окремий прапорець сторно з посиланням на початковий запис, єдину специфікацію `extra` для всіх постачальників, обов'язковий реальний час висадки, перевірку меж відстані й часу під час приймання, а також стабільний ідентифікатор поїздки.

**ML-задача.** Природна задача тут regression: тривалість або вартість поїздки на момент подачі. Дані, агреговані за зоною й годиною, також підходять для forecasting попиту.